# Customer Churn Prediction – Reproducible Pipeline
**Project:** A Reproducible and Fairness-Aware Machine Learning Pipeline for Customer Churn Prediction in a Subscription Service  
**Author:** Valerio Gomez  
**Date:** June 2026  
**Course:** UNMSM – Research Methods & Scientific Integrity in AI

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings('ignore')

# --- Reproducibility ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print('Environment set. SEED =', SEED)

In [ ]:
# Load data – adjust path if needed
df = pd.read_csv('../data/churn_dataset.csv')
print(df.head())
print(df.info())

In [ ]:
# Minimal EDA
sns.countplot(x='Churn', data=df)
plt.title('Target Variable Distribution (Churn)')
plt.show()
print(df.describe())

In [ ]:
# Preprocessing: split FIRST, then scale
X = df.drop(['CustomerID', 'Churn'], axis=1)
y = df['Churn']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Scale on training data ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

In [ ]:
# Define models and train with MLflow tracking
models = {
    'Logistic Regression': LogisticRegression(random_state=SEED, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=SEED),
    'Random Forest': RandomForestClassifier(random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(random_state=SEED),
    'SVM': SVC(random_state=SEED)
}

mlflow.set_experiment('Churn_Experiment')

results = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        acc = accuracy_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)  # Recall for churn class (1)
        f1  = f1_score(y_test, y_pred)

        mlflow.log_param('model', name)
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('recall', rec)
        mlflow.log_metric('f1', f1)
        mlflow.sklearn.log_model(model, name.replace(' ', '_'))

        results.append({'Model': name, 'Accuracy': acc, 'Recall': rec, 'F1': f1})
        print(f'{name}: Acc={acc:.4f}, Recall={rec:.4f}, F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('Recall', ascending=False)
print('\n--- Ranked by Recall ---')
print(results_df)

## DVC Tracking (run in terminal, not here)
```bash
dvc add ../data/churn_dataset.csv
git add ../data/churn_dataset.csv.dvc ../data/.gitignore
git commit -m "Track churn dataset with DVC"
```

In [ ]:
# View MLflow results
print("MLflow UI: run 'mlflow ui' in terminal, then open http://localhost:5000")
print("\nBest model by Recall:")
print(results_df.iloc[0])